# Narrow vs Wide Transformations: Shuffles and Stage Boundaries

**Module**: 04_execution_engine  
**Notebook**: 06_narrow_and_wide_transformations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Categorizing transformations as Narrow (map, filter) or Wide (groupBy, join, distinct, repartition) and observing network shuffles.
- **Why does this topic exist?**: Wide transformations dictate execution time and network I/O in distributed systems.
- **What problem does it solve?**: Unnecessary wide transformations destroy query scalability.
- **Where is it used in real-world Data Engineering?**: ETL pipeline performance optimization.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Jobs/stages/tasks, partitions |
| **Required depth** | Working knowledge: can classify transformations and predict stage boundaries |
| **Data Engineer** | Classify narrow/wide transformations and eliminate unnecessary shuffles. |
| **Senior Data Engineer** | Choose join/aggregation strategies that reduce data movement. |
| **Spark specialist** | Explain shuffle write/read, pipelining, and stage boundaries. |
| **Next topic** | `04_execution_engine/07_dependencies_and_stage_boundaries.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Narrow transformations can be pipelined in one stage; wide transformations require a shuffle. That single
distinction explains most stage boundaries and much of Spark's network/disk cost. A `groupBy`, `join`, `distinct`,
or `repartition` is not just an API call; it is a data redistribution contract.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Narrow** | Parent partition feeds one child | `map`, `filter`, `select`, `union` |
| **Wide** | Parent partitions feed multiple children | `groupBy`, `join`, `distinct`, `repartition` |
| **Pipelining** | Fuse narrow operators | No materialization between them |
| **Shuffle write** | Map-side output | Partitioned/sorted blocks |
| **Shuffle read** | Reduce-side fetch | Network + disk |
| **Stage boundary** | Where a shuffle occurs | New stage and task set |

### How it works, step by step

1. Narrow transformations keep the partition mapping.
2. The scheduler pipelines them within a stage.
3. A wide transformation inserts an exchange.
4. Map tasks write shuffle blocks; reduce tasks fetch them.
5. The next stage starts after the shuffle map outputs are available.
6. AQE may coalesce reduce partitions or split skewed partitions.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Narrow (1:1 partition) vs Wide (1:N or N:M shuffle)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Narrow_Wide_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("=== Narrow Plan (No Exchange) ===")
orders.filter("quantity > 2").select("order_id").explain()

print("\n=== Wide Plan (Contains Exchange) ===")
orders.groupBy("status").count().explain()

---

## Experiment: Exchange Node Detection

- **1. What are we testing?**: Look for Exchange in physical plan.
- **2. What do we expect?**: Present only in wide transformations.

In [ ]:
plan = orders.distinct()._jdf.queryExecution().executedPlan().toString(); print('Contains Exchange?:', 'Exchange' in plan)

**3. What happened?**: Exchange node present.  
**4. Why did it happen?**: distinct() requires a shuffle to deduplicate across partitions.  
**5. Practical Engineering Lesson**: Exchange in physical plan = network shuffle = stage boundary.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Reducing shuffle cost in an ETL pipeline

- **Industry**: Retail
- **Scenario**: A pipeline performs several `repartition` calls and joins before aggregating.
- **Problem**: Each wide operation writes and reads shuffle data, dominating runtime.
- **Why this concept is relevant**: Removing unnecessary repartitions and filtering before joins can cut shuffle volume.
- **Implementation idea**: Inspect the plan for Exchange nodes, remove redundant repartitions, and filter/project early.
- **Expected result**: Fewer stages and less network/disk I/O.
- **Alternative**: A warehouse with automatic distribution.
- **Trade-offs**: Spark gives explicit control; a warehouse hides distribution but may cost more at scale.

### Use Case 2: Choosing a join strategy

- **Industry**: Telecommunications
- **Scenario**: A large call-record table joins a small tower dimension.
- **Problem**: Sort-merge join shuffles both sides and creates a bottleneck.
- **Why this concept is relevant**: A broadcast join avoids shuffling the large side because the dimension is small.
- **Implementation idea**: Broadcast the dimension, verify the plan, and monitor executor memory.
- **Expected result**: A one-stage join with lower network traffic.
- **Alternative**: A colocated/bucketed join.
- **Trade-offs**: Broadcast is simple but memory-bound; bucketing avoids repeated shuffles for recurring joins.

---

## Business Value

- Lower shuffle volume can reduce runtime and infrastructure cost.
- Fewer stages simplify debugging and retries.
- Better join choices improve SLA predictability.
- Clear classification helps code review.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Filter/project before wide transformations.
- Avoid unnecessary `repartition`/`distinct`.
- Broadcast only small dimensions.
- Monitor shuffle read/write and spill.
- Use AQE where appropriate, but verify the final plan.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Classify and verify

**Steps**

1. Build a pipeline with `filter`, `select`, `groupBy`, and `join`.
2. Predict which operations are narrow/wide.
3. Run the job and count stages in the UI.

**What to observe**: Each wide operation adds an exchange and a stage boundary.

**Healthy result**: Your prediction matches the UI.

**Problematic result**: A hidden `distinct` or `repartition` adds an unexpected shuffle.

**Engineering takeaway**: Read the plan for Exchange nodes.

---

## Trade-offs and Alternatives

### Advantages

- Clear performance model
- Enables targeted optimization
- Maps to stage boundaries

### Limitations

- Some operations are wide by nature
- AQE changes runtime behavior
- Shuffle cost depends on data size/skew

### When to use it

- Pipeline design
- Join tuning
- Stage debugging

### When not to use it

- Unnecessary repartition
- Assuming all joins shuffle
- Ignoring shuffle metrics

### Alternatives inside Spark

- Broadcast joins
- AQE
- Bucketing

### Alternatives outside Spark

- Warehouse distribution
- Streaming joins

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Classify Filter
**Problem Statement**: Is filter narrow or wide?  
**Input / Context**: filter

In [ ]:
# B

### Exercise 2 (Intermediate): Classify Distinct
**Problem Statement**: Is distinct narrow or wide?  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Pipeline Optimization
**Problem Statement**: Filter before wide groupBy.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Production Debugging Exercise

### The unnecessary repartition

**Scenario**: A pipeline repartitions three times before a `groupBy` and runs slowly.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Inspect Exchange nodes in the plan.
2. Remove repartitions that do not change the join/aggregation key.
3. Use `coalesce` only when reducing partitions without redistribution.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Classify Filter):
```python
print('Narrow')
```

#### Solution 2 (Classify Distinct):
```python
print('Wide (Exchange node)')
```

#### Solution 3 (Pipeline Optimization):
```python
orders.filter('quantity > 5').groupBy('channel').count().show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Narrow**: In-memory pipelining, no network transfer.
- **Wide**: Network shuffle, writes shuffle files to disk.

### Essential Technical Terms
- `Exchange`: Physical operator executing shuffle.
- `Pipelining`: Chaining narrow steps.

### Common Engineering Mistakes
- **Assuming distinct() is free**: distinct() requires a full network shuffle.

### Interview Questions on This Topic
- **Q: Why is narrow faster than wide?**
  - *A*: Narrow does not write to disk or transfer data over the network.


### More Interview Questions

- **Q: What is the difference between a narrow and a wide transformation?**
  - *A*: Narrow transformations do not require data movement between partitions; wide transformations require a shuffle because multiple child partitions depend on the same parent partition.
- **Q: Why is a broadcast join often faster than a sort-merge join?**
  - *A*: It avoids shuffling the large side by sending a small table to every executor, at the cost of memory on each executor.

### What to Study Next
Open 07_dependencies_and_stage_boundaries.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')